[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/en_clase/S05_03_gradiente_regresion_en_clase.ipynb)

# S05_03 · Descenso del gradiente y regresión desde cero (en clase)

<h2 align="center">Enseñar a una recta a ajustarse sola a los datos</h2>

| | |
|---|---|
| ⏱ **Cuándo** | Jueves · unos 45 min |
| 👥 **Cómo** | En parejas: una persona escribe, la otra lee y comprueba (cambiad de papel al empezar la parte B) |
| 🧰 **Usáis** | 🌐 Google Colab (este cuaderno) · no hay que instalar nada |
| 🏁 **Al terminar** | Ejercicio 4 (el gradiente) y ejercicio 5 (entrenar) con todo ✅ en el paso 6 |

**Este cuaderno tiene dos partes:**

- **1 · Guía paso a paso**: solo lo que hay que hacer. Va en dos bloques: **A · El gradiente** y **B · Entrenar**.
- **2 · Explicaciones**: el porqué de cada cosa, para cuando lo necesitéis (está al final, también en dos bloques: A y B).
- Si queréis, seguid solo la parte 1.

**Solo trae los dos ejercicios de clase**: el **4 trata del gradiente** (parte A) y el **5 de entrenar** (parte B). Los demás (1, 2, 3, 6, 7 y el mini-reto) están en el cuaderno completo [`S05_03_gradiente_regresion`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_03_gradiente_regresion.ipynb), para casa.

## Cómo se usa este cuaderno (leedlo una vez)

**Un cuaderno es una mezcla de textos y celdas de código.** Las celdas de código tienen a la izquierda un botón **▶**.

1. **Ejecutar una celda**
   - Pulsad el botón **▶** de la izquierda, o haced clic dentro de la celda y pulsad **Mayús + Intro**.
   - Debajo de la celda aparece **lo que ha salido**.
2. **De arriba abajo, en orden**
   - Cada celda usa lo que han hecho las anteriores.
   - Si os saltáis una, saldrá un error rojo que dice `NameError`. No pasa nada: ejecutad las que faltan y repetid.
3. **Las celdas con `...`**
   - Son los ejercicios. Los **tres puntos `...` se sustituyen** por vuestro código.
   - No borréis el resto de la celda.
4. **Cada paso termina en ✅**
   - Es lo que **tiene que salir**.
   - Los decimales lejanos pueden cambiar un poco en vuestro ordenador; lo demás tiene que ser **igual**.
   - Si no sale, no sigáis: mirad la parte 2, apartado «Si os atascáis».
5. **Si Colab se reinicia o cerráis la pestaña**
   - Se pierde lo ejecutado, no lo escrito.
   - Volved a ejecutar las celdas **desde el paso 0 hasta donde estabais**.

# 1 · Guía paso a paso

**Qué vais a hacer**

- **Parte A · El gradiente (ejercicio 4):** prepararéis los datos de **1096 días** y escribiréis la función que dice **hacia dónde hay que mover la recta** para que se equivoque menos.
- **Parte B · Entrenar (ejercicio 5):** repetiréis ese movimiento **200 veces** y la recta acabará ajustada a los datos **sin que nadie la dibuje**.
- Es el mismo mecanismo con el que aprenden las redes neuronales.

## Antes de empezar · Paso 0 · Preparar el cuaderno (3 min)

| # | Qué hacéis |
|---|---|
| **0.1** | Ejecutad la celda de abajo (la de «Preparación»). Tarda unos segundos: busca el fichero de datos y, si no está, lo descarga. |
| **0.2** | Ejecutad la celda siguiente (la de las librerías). |

✅ Con la primera sale `Preparación lista ✅` y con la segunda `Librerías listas ✅`.

*No hace falta entender estas dos celdas: dejan preparado lo que usaremos después.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

RESULTADOS = {}

def comprobar(nombre, condicion, ayuda):
    """Escribe ✅ si la condición es cierta y ❌ con una pista si no. No hace falta entender cómo funciona."""
    try:
        ok = bool(condicion())
    except Exception:
        ok = False
    RESULTADOS[nombre] = ok
    print(("✅ " + nombre) if ok else ("❌ " + nombre + " → " + ayuda))

dato("ocupacion_diaria.csv")
print("Preparación lista ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Librerías listas ✅")

# Parte A · El gradiente (ejercicio 4) · unos 20 min

**Qué vais a hacer**

- Vais a ajustar una **recta** que relacione la **temperatura** con la **ocupación**: `ocupación ≈ w0 + w1 × temperatura`.
- `w0` y `w1` son dos números que **hay que encontrar**. Se llaman **parámetros**.
- Para encontrarlos hace falta saber dos cosas: **cuánto se equivoca** la recta (el error) y **hacia dónde mover** `w0` y `w1` para que se equivoque menos (el **gradiente**).
- El **ejercicio 4** es escribir la función que calcula ese gradiente.
- La parte A **no entrena** nada todavía: eso es la parte B.

## Paso 1 · Cargar los datos y prepararlos (5 min)

**Los datos**

- Los mismos **1096 días** de un alojamiento de Tenerife del cuaderno S05_01.
- Hoy usamos dos columnas: la **ocupación** (%) y la **temperatura** (°C).

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **1.1** | Ejecutad la celda 1.1 (carga la tabla y guarda `ocupacion` y `temperatura`). | `1096 días` |
| **1.2** | Ejecutad la celda 1.2 (estandariza la temperatura, como hicisteis en el S05_01). | `[-0.92 -1.51 -1.45]` |
| **1.3** | Ejecutad la celda 1.3 (monta la matriz `X` y guarda la ocupación en `y`). | `(1096, 2)` y dos filas que empiezan por `1.` |

- `X` tiene **dos columnas**: una de **unos** (para `w0`) y otra con la **temperatura estandarizada** (para `w1`).
- `y` es lo que queremos **predecir**: la ocupación.

In [ ]:
# 1.1 · Cargar la tabla
tabla = np.genfromtxt(dato("ocupacion_diaria.csv"), delimiter=",", skip_header=1, usecols=(1, 2, 3, 4))
ocupacion = tabla[:, 0]
temperatura = tabla[:, 1]
print(len(ocupacion), "días")

In [ ]:
# 1.2 · Estandarizar la temperatura (restar la media y dividir entre la desviación)
media_t = temperatura.mean()
desv_t = temperatura.std()
temperatura_z = (temperatura - media_t) / desv_t
print(temperatura_z[:3].round(2))

In [ ]:
# 1.3 · La matriz X (una columna de unos y la temperatura estandarizada) y lo que queremos predecir, y
X = np.column_stack([np.ones(len(temperatura_z)), temperatura_z])
y = ocupacion
print(X.shape)
print(X[:2].round(2))

## Paso 2 · Medir cuánto se equivoca la recta (5 min)

**Objetivo**

- Ver con un ejemplo qué es el **error** (el MSE) y cómo se calcula.
- Ver qué predice una recta con unos `w0` y `w1` dados.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **2.1** | Ejecutad la celda 2.1: define dos funciones ya hechas (`mse` y `derivada_numerica`) y calcula el error de **dos días** a mano. | `Error de 2 días: 100.0` |
| **2.2** | Ejecutad la celda 2.2: la recta con `w0 = 50` y `w1 = 3` predice la ocupación de los 3 primeros días. | `[47.23 45.47 45.66]` |
| **2.3** | Ejecutad la celda 2.3: compara el error de dos rectas (la que predice siempre 0 y la de 2.2). | `Error con w = [0, 0]: 4782.99` y `Error con w = [50, 3]: 479.38` |

- **El MSE es el «error medio»**: cuanto **más pequeño**, mejor ajustada está la recta.
- En 2.1: reales `70` y `50`, predichos `60` y `60`. Errores `+10` y `−10`, al cuadrado `100` y `100`, media `100`.
- `derivada_numerica` no hace falta entenderla: solo la usa la comprobación del ejercicio 4.

In [ ]:
# 2.1 · Funciones ya hechas (no hay que cambiar nada)
def mse(y_real, y_pred):
    """Error cuadrático medio: la media de (real - predicho) al cuadrado."""
    return np.mean((y_real - y_pred) ** 2)

def derivada_numerica(func, x, h=1e-6):
    """Aproxima la pendiente de func en x (solo la usa la comprobación del ejercicio 4)."""
    return (func(x + h) - func(x - h)) / (2 * h)

# El error de dos días, a mano: reales 70 y 50, la recta predice 60 y 60
print("Error de 2 días:", mse(np.array([70, 50]), np.array([60, 60])))

In [ ]:
# 2.2 · Qué predice una recta: y_pred = X @ w   (w0 = 50, w1 = 3)
w_prueba = np.array([50.0, 3.0])
y_pred = X @ w_prueba
print(y_pred[:3].round(2))

In [ ]:
# 2.3 · Comparar dos rectas: cuanto más pequeño el error, mejor
print("Error con w = [0, 0]:", round(mse(y, X @ np.array([0.0, 0.0])), 2))
print("Error con w = [50, 3]:", round(mse(y, X @ w_prueba), 2))

## Paso 3 · Ejercicio 4 · El gradiente (10 min) 🏫

**Objetivo**

- Escribir una función que diga, para `w0` y `w1`, **hacia dónde moverlos** para que el error baje.
- Esa indicación es el **gradiente**: un número por cada parámetro.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **3.1** | **A mano, con 2 días.** Ejecutad la celda 3.1: hace el cálculo del gradiente con dos días y `w = [0, 0]`. | `Errores: [-70. -50.]` y `Gradiente: [-120.   20.]` |
| **3.2** | **Vuestro turno.** En la celda 3.2, sustituid los dos `...` por **las mismas dos líneas** de la 3.1, pero con `X`, `y` y `w` en lugar de `X_dos`, `y_dos` y `w_cero`. | No sale nada |
| **3.3** | Ejecutad la celda 3.3 (la comprobación). | Tres líneas con ✅ |
| **3.4** | Ejecutad la celda 3.4 (el gradiente con los datos reales y `w = [0, 0]`). | `[-136.98   12.07]` |

- **Pista para 3.2:**
  - `errores = X @ w - y`
  - `return 2 / len(y) * X.T @ errores`
- Los números de 3.1 se leen así:
  - **Negativo** (`−120`): hay que **subir** ese parámetro.
  - **Positivo** (`20`): hay que **bajarlo**.
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.

In [ ]:
# 3.1 · A mano, con dos días: uno frío (z = -1, ocupación 70) y uno cálido (z = +1, ocupación 50)
X_dos = np.array([[1.0, -1.0],
                  [1.0,  1.0]])
y_dos = np.array([70.0, 50.0])
w_cero = np.array([0.0, 0.0])

errores = X_dos @ w_cero - y_dos                      # lo que predice la recta menos lo real
gradiente = 2 / len(y_dos) * X_dos.T @ errores        # 2 / n por X.T @ errores
print("Errores:", errores)
print("Gradiente:", gradiente)

In [ ]:
# 3.2 · TU TURNO: las mismas dos líneas de la celda 3.1, pero con X, y y w
def gradiente_mse(X, y, w):
    errores = ...            # lo que predice la recta menos lo real: X @ w - y
    return ...               # 2 / len(y) * X.T @ errores

In [ ]:
# 3.3 · Comprobación del ejercicio 4 (no hay que cambiar nada)
numerico = np.array([
    derivada_numerica(lambda a: mse(y, X @ np.array([a, w_prueba[1]])), w_prueba[0]),
    derivada_numerica(lambda b: mse(y, X @ np.array([w_prueba[0], b])), w_prueba[1]),
])
comprobar("gradiente: un valor por parámetro", lambda: gradiente_mse(X, y, w_prueba).shape == (2,), "debe devolver 2 números, uno para w0 y otro para w1. ¿Has cambiado los puntos ... por tu código?")
comprobar("gradiente: ejemplo de 2 días", lambda: np.allclose(gradiente_mse(X_dos, y_dos, w_cero), [-120.0, 20.0]), "con los 2 días de la celda 3.1 debe salir [-120, 20]: errores = X @ w - y y 2 / len(y) * X.T @ errores")
comprobar("gradiente: coincide con el numérico", lambda: np.allclose(gradiente_mse(X, y, w_prueba), numerico, rtol=1e-3), "revisa la fórmula: 2 / len(y) * X.T @ errores, con errores = X @ w - y")

In [ ]:
# 3.4 · Mirar el resultado: el gradiente con los datos reales y w = [0, 0]
print(gradiente_mse(X, y, np.zeros(2)).round(2))

✅ Tiene que salir: `✅ gradiente: un valor por parámetro`, `✅ gradiente: ejemplo de 2 días` y `✅ gradiente: coincide con el numérico`. Si falta alguno, no sigáis.

# Parte B · Entrenar (ejercicio 5) · unos 20 min

*Cambiad de papel: quien escribía, ahora lee y comprueba.*

**Qué es entrenar y qué vais a hacer**

- **Entrenar** es repetir muchas veces: **calcular el gradiente** y **dar un pasito** en la dirección que baja el error.
- Primero **veréis un solo paso** con vuestra función y cómo baja el error (paso 4).
- Después, el **ejercicio 5**: poner ese paso dentro de un bucle de 200 vueltas (paso 5).
- Para esta parte se usa la función `gradiente_mse` de la parte A: si no la tenéis, volved al paso 3.

## Paso 4 · Dar un paso cuesta abajo (7 min) 🏫

**Objetivo**

- Ver que un paso en la dirección que dice el gradiente **hace bajar el error**.
- Ver que repetir el paso lo sigue bajando.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **4.1** | Ejecutad la celda 4.1: da **un paso** desde `w = [0, 0]` con la tasa `0.1`. | `[13.7  -1.21]` |
| **4.2** | Ejecutad la celda 4.2: compara el error antes y después del paso. | `Antes: 4782.99 · Después: 3081.25` |
| **4.3** | Ejecutad la celda 4.3: repite el paso **3 veces** con un bucle `for`. | Tres líneas, con el error cada vez más pequeño: `3081.3`, `1992.1` y `1295.1` |

- **La regla de cada paso:** `w = w - tasa * gradiente`.
- La **tasa** (`0.1`) es el tamaño del paso: pequeña = lento; muy grande = nos pasamos y el error sube.
- El bucle de 4.3 es casi lo que vais a escribir en el ejercicio 5.

In [ ]:
# 4.1 · Un solo paso cuesta abajo, desde w = [0, 0]
w = np.zeros(2)
tasa = 0.1
gradiente = gradiente_mse(X, y, w)
w_nuevo = w - tasa * gradiente
print(w_nuevo.round(2))

In [ ]:
# 4.2 · El error antes y después del paso
print(f"Antes: {mse(y, X @ w):.2f} · Después: {mse(y, X @ w_nuevo):.2f}")

In [ ]:
# 4.3 · Repetir el paso 3 veces
w = np.zeros(2)
for paso in range(1, 4):
    w = w - 0.1 * gradiente_mse(X, y, w)
    print(f"Paso {paso}: w = {w.round(2)} · error = {mse(y, X @ w):.1f}")

## Paso 5 · Ejercicio 5 · Entrenar (10 min) 🏫

**Objetivo**

- Escribir la función `entrenar`, que repite el paso **200 veces** y se queda con el último `w`.
- Guardar el error de cada vuelta en una lista (`historial`) para ver cómo va bajando.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **5.1** | **Vuestro turno.** En la celda 5.1, sustituid los **tres `...`**. Los dos primeros son las líneas de la celda 4.3; el tercero guarda el error de esa vuelta. | No sale nada |
| **5.2** | Ejecutad la celda 5.2 (entrena de verdad con 200 vueltas). | `Error al empezar: 3081.3 · al terminar: 55.95` y `w final: [68.49 -6.03]` |
| **5.3** | Ejecutad la celda 5.3 (la comprobación). | Cuatro líneas con ✅ |
| **5.4** | Ejecutad la celda 5.4 (dibuja cómo baja el error y la recta que ha encontrado). | Dos gráficos: una curva que baja y una recta que atraviesa la nube de puntos |
| **5.5** | Ejecutad la celda 5.5 (traduce la recta a «puntos por grado»). | `ocupación ≈ 107.73 + (-1.86) × temperatura` |

- **Pistas para 5.1:**
  - `gradiente = gradiente_mse(X, y, w)`
  - `w = w - tasa * gradiente`
  - `historial.append(mse(y, X @ w))`
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.
- El resultado de 5.5 dice: **por cada grado más, la ocupación baja unos 1,86 puntos**. Lo ha encontrado vuestro código, sin ninguna librería.

In [ ]:
# 5.1 · TU TURNO: entrenar = repetir "calcular el gradiente y dar un paso cuesta abajo"
def entrenar(X, y, tasa=0.1, epocas=200):
    w = np.zeros(X.shape[1])             # empezamos con todos los parámetros a 0
    historial = []                       # aquí guardamos el error de cada vuelta
    for epoca in range(epocas):
        gradiente = ...                  # el gradiente ahora: gradiente_mse(X, y, w)
        w = ...                          # un paso cuesta abajo: w - tasa * gradiente
        historial.append(...)            # el error de esta vuelta: mse(y, X @ w)
    return w, historial

In [ ]:
# 5.2 · Entrenar de verdad: 200 vueltas
w_z, historial = entrenar(X, y)
print(f"Error al empezar: {historial[0]:.1f} · al terminar: {historial[-1]:.2f}")
print("w final:", w_z.round(2))

In [ ]:
# 5.3 · Comprobación del ejercicio 5 (no hay que cambiar nada)
w_exacto = np.linalg.solve(X.T @ X, X.T @ y)          # la solución exacta, para comparar
comprobar("entrenar: un error por vuelta", lambda: len(historial) == 200 and all(isinstance(h, (int, float, np.floating)) for h in historial), "el historial debe tener 200 valores: añade un error en cada vuelta con historial.append(...). ¿Has cambiado los puntos ... por tu código?")
comprobar("entrenar: el error baja", lambda: historial[-1] < historial[0], "el error final debe ser menor que el primero: w = w - tasa * gradiente (con el signo menos)")
comprobar("entrenar: el error nunca sube", lambda: all(a >= b - 1e-9 for a, b in zip(historial, historial[1:])), "en cada vuelta el error debe bajar o quedarse igual: ¿restas el gradiente (w - tasa * gradiente)?")
comprobar("entrenar: llega a la solución exacta", lambda: np.allclose(w_z, w_exacto, atol=1e-3), "w final debe salir [68.49 -6.03]: revisa que el gradiente se calcula con el w de cada vuelta")

In [ ]:
# 5.4 · Ver la curva de aprendizaje y la recta ajustada
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ejes[0].plot(historial)
ejes[0].set_xlabel("Vuelta (época)")
ejes[0].set_ylabel("Error (MSE)")
ejes[0].set_title("El error baja en cada vuelta")

ejes[1].scatter(temperatura, ocupacion, s=6, alpha=0.4, label="Días")
t_lin = np.linspace(temperatura.min(), temperatura.max(), 50)
ejes[1].plot(t_lin, w_z[0] + w_z[1] * (t_lin - media_t) / desv_t, color="crimson", label="Recta encontrada")
ejes[1].set_xlabel("Temperatura (°C)")
ejes[1].set_ylabel("Ocupación (%)")
ejes[1].set_title("Ocupación frente a temperatura")
ejes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 · Volver a "puntos por grado" (deshacer la estandarización)
pendiente = w_z[1] / desv_t
intercepto = w_z[0] - w_z[1] * media_t / desv_t
print(f"ocupación ≈ {intercepto:.2f} + ({pendiente:.2f}) × temperatura")

## Paso 6 · Cerrar los dos ejercicios (2 min)

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **6.1** | Ejecutad la celda 6.1 (cuenta las comprobaciones ✅ de cada ejercicio por separado). | `Parte A · El gradiente (ejercicio 4): 3 de 3 ✅`, `Parte B · Entrenar (ejercicio 5): 4 de 4 ✅` y `🎉 Los ejercicios 4 y 5 están hechos` |
| **6.2** | Si os sobra tiempo, abrid el cuaderno completo [`S05_03_gradiente_regresion`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_03_gradiente_regresion.ipynb) y probad los ejercicios 6 (la tasa de aprendizaje) y 7 (comparar con scikit-learn). | — |

In [ ]:
# 6.1 · Resumen por ejercicio
N4 = ["gradiente: un valor por parámetro", "gradiente: ejemplo de 2 días", "gradiente: coincide con el numérico"]
N5 = ["entrenar: un error por vuelta", "entrenar: el error baja", "entrenar: el error nunca sube", "entrenar: llega a la solución exacta"]
bien4 = sum(RESULTADOS.get(n, False) for n in N4)
bien5 = sum(RESULTADOS.get(n, False) for n in N5)
print(f"Parte A · El gradiente (ejercicio 4): {bien4} de 3 ✅")
print(f"Parte B · Entrenar (ejercicio 5): {bien5} de 4 ✅")
if bien4 == 3 and bien5 == 4:
    print("🎉 Los ejercicios 4 y 5 están hechos")
else:
    print("Faltan comprobaciones: mirad las líneas ❌ de las celdas 3.3 (ejercicio 4) y 5.3 (ejercicio 5)")

## ✅ Terminado cuando

**Parte A · El gradiente (ejercicio 4)**

- [ ] La celda 3.3 muestra las tres líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué dice el gradiente** (hacia dónde mover cada parámetro).

**Parte B · Entrenar (ejercicio 5)**

- [ ] La celda 5.3 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué se repite en cada vuelta** del entrenamiento.

**Los dos**

- [ ] La celda 6.1 dice `🎉 Los ejercicios 4 y 5 están hechos`.

---

# 2 · Explicaciones

*Para cuando una cosa no sale o queréis entender por qué se hace así. No hace falta leerlo para resolver los ejercicios. Está en dos bloques, igual que la guía: **A · El gradiente** y **B · Entrenar**.*

# Explicaciones de la parte A · El gradiente (ejercicio 4)

## La idea: un modelo es una fórmula con números ajustables

- Un **modelo** es una fórmula que **predice** algo. Aquí: `ocupación ≈ w0 + w1 × temperatura`.
- Los números `w0` y `w1` son los **parámetros**:
  - **`w0`**: el punto de partida (la ocupación cuando la temperatura estandarizada vale 0, es decir, la media).
  - **`w1`**: cuánto cambia la ocupación al subir la temperatura una desviación.
- **Entrenar** es buscar los `w0` y `w1` con los que el modelo **se equivoca menos**.
- Es lo mismo que hace una red neuronal, pero con muchísimos más parámetros.

## Qué es la matriz `X` y qué hace `X @ w`

- `X` tiene **una fila por día** y **dos columnas**:

| Columna de `X` | Qué contiene | Para qué sirve |
|---|---|---|
| **0** | Siempre `1` | Multiplica a `w0` (así `w0` entra siempre en la suma) |
| **1** | La temperatura estandarizada de ese día | Multiplica a `w1` |

- `X @ w` hace, **para cada día**, esta cuenta: `1 × w0 + temperatura_z × w1`. Es la **predicción** de la recta.
- Por ejemplo, el primer día, con `w = [50, 3]`: `1 × 50 + (−0.92) × 3 = 47.23`.
- El símbolo `@` es el **producto de matrices**: «multiplicar y sumar» fila por columna. No hace falta saber más.
- La **temperatura estandarizada** se calcula como en el S05_01: restar la media y dividir entre la desviación. Así `w0` y `w1` tienen escalas parecidas y el entrenamiento va mucho mejor.

## El error: el MSE

- **MSE** son las siglas en inglés de **error cuadrático medio**. Se calcula en tres pasos:
  1. Para cada día: **real − predicho** (el error de ese día).
  2. Elevar cada error **al cuadrado**.
  3. Hacer la **media**.
- Con dos días (reales `70` y `50`, predichos `60` y `60`):

| Día | Real | Predicho | Error | Error al cuadrado |
|---|---|---|---|---|
| 1 | 70 | 60 | +10 | 100 |
| 2 | 50 | 60 | −10 | 100 |

- Media de `100` y `100` = **100**. Ese es el MSE.
- **Cuanto más pequeño, mejor ajustada** está la recta.

## El gradiente: hacia dónde mover cada parámetro

- Imaginad que estáis en una montaña **con niebla** y queréis bajar al valle:
  - No veis el valle, pero **notáis la pendiente del suelo** bajo los pies.
  - Si el suelo baja hacia la derecha, dais un paso a la derecha. Y así, paso a paso.
- Aquí la «montaña» es el **error**. El «suelo» depende de `w0` y `w1`.
- El **gradiente** dice, **para cada parámetro**, cuánto **sube el error** si lo aumentamos un poco:
  - **Gradiente negativo:** al aumentar el parámetro el error **baja** → hay que **subirlo**.
  - **Gradiente positivo:** al aumentar el parámetro el error **sube** → hay que **bajarlo**.
- En 3.1 sale `[-120, 20]`: `w0` hay que subirlo y `w1` hay que bajarlo.

## La fórmula del gradiente, leída en partes

La fórmula es `2 / n × Xᵀ (X w − y)`. En código: `2 / len(y) * X.T @ errores`.

- **`errores = X @ w - y`:** lo que predice la recta menos lo real, **día a día**.
- **`X.T @ errores`:** para **cada parámetro**, suma de «error × el valor de su columna»:
  - Para `w0` (columna de unos): la **suma de los errores**.
  - Para `w1` (columna de temperaturas): la suma de «error × temperatura».
- **`2 / len(y)`:** divide entre el número de días (es una media) y multiplica por 2 (el 2 sale de derivar el cuadrado).
- `.T` «gira» la tabla: lo que eran columnas pasan a ser filas.

Con los dos días de la celda 3.1 (`w = [0, 0]`):

| Cuenta | Para `w0` | Para `w1` |
|---|---|---|
| Errores | −70 y −50 | −70 y −50 |
| Error × valor de su columna | (−70 × 1) + (−50 × 1) = **−120** | (−70 × −1) + (−50 × 1) = **20** |
| × 2 / 2 | **−120** | **20** |

## Las palabras de la parte A

| Palabra | Qué significa |
|---|---|
| **Modelo** | Una fórmula que predice algo (aquí, la ocupación) |
| **Parámetros (`w0`, `w1`)** | Los números ajustables del modelo |
| **Predicción (`y_pred`)** | Lo que dice el modelo para cada día |
| **Error** | Real menos predicho |
| **MSE** | La media de los errores al cuadrado: el «error medio» |
| **Gradiente** | Para cada parámetro, hacia dónde moverlo para que el error baje |
| **`X @ w`** | Multiplicar y sumar: la predicción de todos los días de una vez |
| **`X.T`** | La tabla `X` girada (filas ↔ columnas) |
| **`def` / `return`** | Crear una función / devolver su resultado |
| **`len(y)`** | Cuántos datos tiene `y` (aquí, 1096) |
| **✅ / ❌** | Lo que escribe la función `comprobar`: ✅ está bien; ❌ hay algo que corregir (con una pista) |

## Por qué se hace así

- **¿Por qué elevar el error al cuadrado?**
  - Para que `+10` y `−10` **no se cancelen** al hacer la media.
  - Para que los errores grandes **pesen más** que los pequeños.
- **¿Por qué una columna de unos en `X`?**
  - Para que la recta tenga **punto de partida** (`w0`). Sin ella, la recta pasaría siempre por el origen.
- **¿Por qué comprobar el gradiente con `derivada_numerica`?**
  - La derivada numérica calcula la pendiente «a lo bruto», probando un cambio minúsculo.
  - Si vuestra fórmula da lo mismo, es que **está bien escrita**.
- **¿Por qué estandarizar la temperatura?**
  - Sin estandarizar, un paso bueno para `w0` es demasiado grande para `w1` (o al revés) y el entrenamiento va lentísimo o se descontrola.

# Explicaciones de la parte B · Entrenar (ejercicio 5)

## La idea: entrenar es repetir un pasito

- En cada **vuelta** (se llama **época**) se hace lo mismo:
  1. **Calcular el gradiente** con los `w` de ahora.
  2. **Dar un paso** en la dirección contraria: `w = w - tasa * gradiente`.
  3. **Apuntar el error** en el `historial`.
- Se empieza con `w = [0, 0]` y se repite 200 veces. El error va bajando hasta casi quedarse quieto.
- Con los números del paso 4:

| Vuelta | `w` | Error |
|---|---|---|
| 0 (al empezar) | [0, 0] | 4782.99 |
| 1 | [13.7, −1.21] | 3081.3 |
| 2 | [24.66, −2.17] | 1992.1 |
| 3 | [33.42, −2.94] | 1295.1 |
| … | … | … |
| 200 | [68.49, −6.03] | 55.95 |

## La tasa de aprendizaje

- La **tasa** es el **tamaño del paso**. Aquí vale `0.1`.

| Tasa | Qué pasa |
|---|---|
| Muy pequeña (`0.005`) | El error baja, pero **muy despacio** |
| Adecuada (`0.1`) | El error baja rápido y se queda quieto |
| Demasiado grande (`1.2`) | Nos **pasamos** del mínimo, el error **sube** y se descontrola |

- Es como bajar una escalera: pasos muy cortos tardan años; saltos enormes te hacen caer.
- Probarlo es el **ejercicio 6** del cuaderno completo (para casa).

## Cómo se leen los gráficos del paso 5.4

- **Curva de la izquierda:** el error en cada vuelta. Baja muy rápido al principio y luego se aplana: **ya no se puede mejorar más**.
- **Gráfico de la derecha:** los puntos son los 1096 días y la línea roja es la recta encontrada.
- La recta baja hacia la derecha: **cuanto más calor, menos ocupación** en este alojamiento.
- En 5.5 la recta se traduce a unidades de verdad: `107.73 − 1.86 × temperatura`.
  - Para pasar de «por desviación» a «por grado» hay que **deshacer la estandarización**: `pendiente = w1 / desviación`.

## Las palabras de la parte B

| Palabra | Qué significa |
|---|---|
| **Entrenar** | Buscar los parámetros que hacen que el error sea lo más pequeño posible |
| **Época** | Una vuelta del entrenamiento (gradiente + paso) |
| **Tasa de aprendizaje (`tasa`)** | El tamaño del paso en cada vuelta |
| **Bucle `for`** | Repetir algo un número de veces |
| **`historial`** | Una lista donde se apunta el error de cada vuelta |
| **`.append(...)`** | Añadir un dato al final de una lista |
| **`np.zeros(2)`** | Un array de dos ceros: `[0, 0]` |
| **Solución exacta** | Lo que sale resolviendo las ecuaciones de golpe (`np.linalg.solve`): sirve para comprobar |
| **Estandarizar / desestandarizar** | Pasar los datos a «desviaciones» / volver a las unidades originales |

## Por qué se hace así

- **¿Por qué se empieza con `w = [0, 0]`?**
  - Es un punto de partida cualquiera. Si el entrenamiento funciona, da igual por dónde se empiece.
- **¿Por qué se resta el gradiente y no se suma?**
  - El gradiente apunta **cuesta arriba** (hacia donde el error sube). Para bajar hay que ir en sentido contrario.
  - Si se suma, el error **sube** y la comprobación «el error baja» da ❌.
- **¿Por qué 200 épocas?**
  - Con esta tasa, a partir de unas 50 vueltas el error casi no cambia. 200 es margen de sobra.
- **¿Por qué comparar con la solución exacta?**
  - Este problema sencillo se puede resolver también de golpe. Si el entrenamiento llega al mismo sitio, es que **funciona**.
  - Con modelos grandes ya no hay solución exacta, y entrenar es la única manera.

## 🚑 Si os atascáis

**Parte A · El gradiente (ejercicio 4)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `❌ gradiente: un valor por parámetro` | La función no devuelve 2 números | Comprobad que el `return` lleva `2 / len(y) * X.T @ errores` |
| `❌ gradiente: ejemplo de 2 días` | La fórmula no es la de la celda 3.1 | Copiad las dos líneas de 3.1 cambiando `X_dos` por `X`, `y_dos` por `y` y `w_cero` por `w` |
| `❌ gradiente: coincide con el numérico` | Falta el `2 /` o el signo de `errores` | `errores = X @ w - y` (predicho **menos** real) y `2 / len(y) * X.T @ errores` |
| `TypeError` que nombra `'ellipsis'` en 3.3 o 3.4 | Se ha quedado algún `...` sin sustituir | Cambiad los `...` de la celda 3.2 por vuestro código y volved a ejecutarla |
| `NameError: name 'w_prueba' is not defined` en 3.3 | No se ejecutó la celda 2.2 | Ejecutad las celdas desde el paso 0 hasta la que falla, en orden |

**Parte B · Entrenar (ejercicio 5)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'gradiente_mse' is not defined` | No se ha hecho el ejercicio 4 | Volved al paso 3 y ejecutad la celda 3.2 |
| `❌ entrenar: un error por vuelta` | Falta el `historial.append(...)` o está fuera del bucle | Una línea `historial.append(mse(y, X @ w))` **dentro** del `for` (con sangría) |
| `❌ entrenar: el error baja` / `el error nunca sube` | Se suma el gradiente en vez de restarlo | `w = w - tasa * gradiente` (signo **menos**) |
| `❌ entrenar: llega a la solución exacta` | El gradiente no se recalcula en cada vuelta | `gradiente = gradiente_mse(X, y, w)` **dentro** del `for` |
| `IndentationError` | El contenido del `for` no está desplazado a la derecha | Las tres líneas del bucle deben tener 4 espacios más que el `for` |
| `TypeError` que nombra `'ellipsis'` en 5.2 | La celda 5.1 todavía tiene los `...` | Sustituidlos y ejecutad 5.1 antes de 5.2 |
| El error sale `nan` o enorme | Una tasa demasiado grande | Dejad `tasa=0.1` (el valor por defecto) |

**En cualquier parte**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'X' is not defined` (o `np`, `y`…) | No habéis ejecutado la celda que crea esa variable, o Colab se reinició | Ejecutad las celdas **desde el paso 0** hasta la que falla, en orden |
| La comprobación da ❌ aunque creéis que está bien | Habéis corregido el código pero no habéis vuelto a ejecutar la celda del ejercicio | Ejecutad la celda del ejercicio **otra vez** y después la de comprobación |
| Sale un error al descargar el fichero (`URLError`, `FileNotFoundError`) | Sin internet o sin acceso a GitHub | Probad otra vez; si sigue, avisad al docente (puede dar el fichero `ocupacion_diaria.csv`) |
| Seguís sin verlo | A veces pasa | 1) Preguntad a vuestra pareja; 2) después, al docente; 3) **cuando hayáis terminado o agotado las pistas**, ejecutad la «Solución de referencia» del ejercicio. Primero intentadlo: lo que copiéis, tenéis que saber explicarlo |

## 🆘 Soluciones de referencia

- Ejecutad estas celdas **solo después de intentarlo**.
- Cada ejercicio tiene la suya.
- Si lo dejáis para después, ejecutadlas para poder seguir con el paso siguiente.

In [ ]:
#@title 🆘 Solución del ejercicio 4 · El gradiente (ábrela solo después de intentarlo)
def gradiente_mse(X, y, w):
    errores = X @ w - y
    return 2 / len(y) * X.T @ errores

print("Solución del ejercicio 4 aplicada: ahora ejecutad la celda 3.3")

In [ ]:
#@title 🆘 Solución del ejercicio 5 · Entrenar (ábrela solo después de intentarlo)
def entrenar(X, y, tasa=0.1, epocas=200):
    w = np.zeros(X.shape[1])
    historial = []
    for epoca in range(epocas):
        gradiente = gradiente_mse(X, y, w)
        w = w - tasa * gradiente
        historial.append(mse(y, X @ w))
    return w, historial

print("Solución del ejercicio 5 aplicada: ahora ejecutad la celda 5.2 y la 5.3")

## Para reflexionar (con vuestra pareja)

**Parte A · El gradiente**

1. En 3.1 el gradiente de `w0` salía `−120`. ¿Por qué hay que **subir** `w0`? *(pensad en si la recta predice por encima o por debajo de los datos)*
2. ¿Qué pasaría con el gradiente si la recta ya fuera la mejor posible?

**Parte B · Entrenar**

1. ¿Por qué la curva de error baja muy deprisa al principio y luego se aplana?
2. Si pusierais `tasa=2.5`, ¿qué esperáis que pase con el error? *(probadlo, o hacedlo en casa con el ejercicio 6)*

## Qué habéis aprendido

**Parte A · El gradiente**

- Un **modelo** es una fórmula con **parámetros** ajustables; el **MSE** mide cuánto se equivoca.
- El **gradiente** dice, para cada parámetro, hacia dónde moverlo para que el error baje.
- Se calcula con una fórmula corta: `2 / len(y) * X.T @ (X @ w - y)`.

**Parte B · Entrenar**

- **Entrenar** = repetir «calcular el gradiente y dar un paso cuesta abajo» (`w = w - tasa * gradiente`).
- La **tasa de aprendizaje** decide el tamaño del paso: ni muy pequeña ni muy grande.
- Vuestro entrenamiento encuentra la misma recta que la solución exacta: no hay magia.